In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

In [17]:
import anthropic
from dotenv import load_dotenv
import pandas as pd 
import importlib
import src.fetcher
import src.filters
importlib.reload(src.fetcher)
importlib.reload(src.filters)

from src.fetcher import configure_entrez, fetch_sra_studies, fetch_runs_for_bioproject
from src.filters import check_hard_filters

load_dotenv('../.env')
configure_entrez()

api_key = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

print('Setup complete. Ready to run tests.')
print('Anthropic client initialized:', client is not None)

Entrez configured with email: akharya@ucsd.edu
Setup complete. Ready to run tests.
Anthropic client initialized: True


In [6]:
# fetch the frog study we validated earlier
frog_ids = fetch_sra_studies("PRJNA836960[BioProject]", max_results=200)
frog_df = None

for sra_id in frog_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is not None and len(runs_df) > 0:
        if frog_df is None:
            frog_df = runs_df
        else:
            frog_df = pd.concat([frog_df, runs_df]).drop_duplicates(subset=["Run"])

filters = check_hard_filters(frog_df)

# summarize study metadata for the agent
study_summary = f"""
BioProject: PRJNA836960
Title: Shotgun versus 16S - Museum and Fresh Leopard Frog Guts
Host organism: Leopard Frog (Amphibia)
Total runs: {len(frog_df)}
Library strategies: {frog_df['LibraryStrategy'].value_counts().to_dict()}
Unique BioSamples: {frog_df['BioSample'].nunique()}
Platform: {frog_df['Platform'].iloc[0]}
Hard filter results: {filters}
"""

print(study_summary)

Found 130 studies. Fetching 130...
 Fetched 100 of 130
 Fetched 130 of 130

BioProject: PRJNA836960
Title: Shotgun versus 16S - Museum and Fresh Leopard Frog Guts
Host organism: Leopard Frog (Amphibia)
Total runs: 130
Library strategies: {'WGS': 112, 'AMPLICON': 18}
Unique BioSamples: 130
Platform: ILLUMINA
Hard filter results: {'has_16S': True, 'has_WGS': True, 'is_paired': True, 'has_fastq': True, 'passes': True}



In [7]:
# first anthropic api call - agent summarizes and assesses a study
message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    system="""You are a microbiome research assistant helping curate studies 
for the Gut Microbiome Tree of Life project (GMToL). GMToL is building 
the most comprehensive cross-species gut microbiome dataset ever assembled, 
spanning insects, fish, amphibians, reptiles, birds, and mammals including 
diverse human populations. The project prioritizes host phylogenetic diversity 
— studies from underrepresented animal clades are more valuable than additional 
human or mouse studies. You help researchers decide which public studies are 
worth including by summarizing study metadata and asking targeted questions.""",
    messages=[
        {
            "role": "user",
            "content": f"""I found a candidate study for GMToL. Please:
1. Summarize what this study is in 2-3 sentences
2. Assess how valuable it is for GMToL and why
3. Ask me one specific question to help decide whether to include it

Here is the study metadata:
{study_summary}"""
        }
    ]
)

print(message.content[0].text)

## Study Summary

This study compares shotgun metagenomic (WGS) and 16S amplicon sequencing approaches on gut samples from leopard frogs, including both museum-preserved and fresh specimens. With 130 samples across both sequencing strategies, it appears to be a methodological comparison study examining how preservation and sequencing method affect microbiome characterization in amphibians.

## Value Assessment for GMToL

**Moderately High Value**

- **Amphibians are underrepresented** in gut microbiome research, making this phylogenetically valuable for GMToL's diversity goals
- **WGS data available** (112 runs) enables deeper taxonomic and functional analysis beyond what 16S provides
- **Paired sequencing approaches** on the same samples could help GMToL understand cross-method comparability
- The inclusion of museum specimens is interesting but may complicate interpretation if preservation affects microbiome profiles

## Key Question

**Were the museum specimens' gut contents preserv

In [10]:
# researcher feedback loop 
print('Question from agent:')
print("Were the museum specimens' gut contents preserved in a way that maintains microbial DNA integrity (e.g., ethanol-preserved vs. formalin-fixed), and do you want to include both museum and fresh samples, or only the fresh specimens to ensure data quality consistency across GMToL?")
print('Do you want to include both museum and fresh samples, or only fresh samples')

#record researcher decision 
decision = input('Include this study? y/n/maybe').strip().lower()
notes = input('Any notes on why? (press enter to skip)').strip()

# store researcher feedback 
decisions = {}
decisions["PRJNA836960"] = {
    "title": "Shotgun versus 16S - Museum and Fresh Leopard Frog Guts",
    "host": "Leopard Frog (Amphibia)",
    "decision": decision,
    "notes": notes,
    "agent_assessment": "Moderately High Value - amphibian, paired data, museum preservation concern"
}
print(f"\nDecision recorded: {decision}")
if notes:
    print(f"Notes: {notes}")

print(f"\nTotal decisions made so far: {len(decisions)}")

Question from agent:
Were the museum specimens' gut contents preserved in a way that maintains microbial DNA integrity (e.g., ethanol-preserved vs. formalin-fixed), and do you want to include both museum and fresh samples, or only the fresh specimens to ensure data quality consistency across GMToL?
Do you want to include both museum and fresh samples, or only fresh samples

Decision recorded: yes
Notes: only fresh samples

Total decisions made so far: 1


Now, trying to make the agent loop through multiple studies at once automatically instead of just one. 

In [12]:
import time 

def run_agent_triage(search_term, max_studies = 5, decisions = None):
    """
    Run the agent triage loop on a set of studies. 
    Fetches studies, filters them based on the hard filters, and presents the ones that pass to the researcher. 

    Args:
    search_term: what to search for in SRA
    max_studies: how many studies to evaluate 
    decisions: existing decisions dictionary to append to

    Returns:
    updated decisions dictionary 
    """

    if decisions is None:
        decisions = {}

    print(f'Searching for: {search_term}\n')
    ids = fetch_sra_studies(search_term, max_results = max_studies * 3)  # fetch extra to account for filtering since most won't pass hard filters

    studies_presented = 0
    
    for sra_id in ids:
        if studies_presented >= max_studies:
            break

        runs_df = fetch_runs_for_bioproject(sra_id)
        if runs_df is None or len(runs_df) ==0:
            continue 
        
        filters = check_hard_filters(runs_df)

        if not filters['passes']:
            continue

        bioproject = runs_df['BioProject'].iloc[0]

        if bioproject in decisions:
            print(f" Skipping {bioproject} - already evaluated")
            continue

        study_summary = f"""
BioProject: {bioproject}
Host organism: {runs_df['ScientificName'].iloc[0]}
Total runs: {len(runs_df)}
Library strategies: {runs_df['LibraryStrategy'].value_counts().to_dict()}
Unique BioSamples: {runs_df['BioSample'].nunique()}
Platform: {runs_df['Platform'].iloc[0]}
Hard filter results: {filters}
"""
        
        # ask agent to assess
        message = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=512,
            system="""You are a microbiome research assistant helping curate 
studies for the Gut Microbiome Tree of Life project (GMToL). GMToL prioritizes 
host phylogenetic diversity — studies from underrepresented animal clades like 
amphibians, reptiles, insects, and fish are more valuable than additional human 
or mouse studies. Be concise — summarize in 2-3 sentences, give a value rating 
(Low/Medium/High/Critical), and ask one specific question.""",
            messages=[
                {
                    "role": "user",
                    "content": f"Assess this candidate study for GMToL:\n{study_summary}"
                }
            ]
        )

        print(f"Study {studies_presented + 1}: {bioproject}")
        print(f"{'='*50}")
        print(message.content[0].text)
        print()
        
        # get researcher feedback
        decision = input("Include this study? (y/n/maybe/stop): ").strip().lower()
        
        if decision == "stop":
            print("Stopping triage session.")
            break
            
        notes = input("Notes (press enter to skip): ").strip()
        
        decisions[bioproject] = {
            "host": runs_df["ScientificName"].iloc[0],
            "decision": decision,
            "notes": notes,
            "strategies": runs_df["LibraryStrategy"].value_counts().to_dict()
        }
        
        studies_presented += 1
        print(f"Decision recorded. ({studies_presented} studies reviewed)\n")
        time.sleep(0.5)
    
    return decisions

# run the agent on paired gut microbiome studies
all_decisions = run_agent_triage(
    search_term="gut microbiome 16S metagenomics",
    max_studies=3,
    decisions=decisions  # pass in existing decisions so we don't repeat
)

print(f"SESSION COMPLETE")
print(f"Total decisions in log: {len(all_decisions)}")
print(f"\nDecision summary:")
for bioproject, info in all_decisions.items():
    print(f"  {bioproject} | {info['host']} | {info['decision']}")

Searching for: gut microbiome 16S metagenomics

Found 91254 studies. Fetching 9...
 Fetched 9 of 9
SESSION COMPLETE
Total decisions in log: 1

Decision summary:
  PRJNA836960 | Leopard Frog (Amphibia) | yes


In [13]:
# try the dual strategy - search for paired studies first
all_decisions = run_agent_triage(
    search_term="16S metagenomics gut paired same samples",
    max_studies=3,
    decisions=all_decisions
)

Searching for: 16S metagenomics gut paired same samples

Found 391 studies. Fetching 9...
 Fetched 9 of 9


In [18]:
# diagnostic - check what's actually in those 9 studies
from Bio import Entrez

handle = Entrez.esearch(
    db="sra",
    term="16S metagenomics gut paired same samples",
    retmax=9
)
record = Entrez.read(handle)
handle.close()

diag_ids = record["IdList"]

for sra_id in diag_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is not None:
        bioproject = runs_df["BioProject"].iloc[0]
        strategies = runs_df["LibraryStrategy"].unique()
        print(f"SRA ID: {sra_id} | BioProject: {bioproject} | Strategies: {strategies} | Runs: {len(runs_df)}")
    time.sleep(0.3)

SRA ID: 36740359 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740358 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740357 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740356 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740355 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740354 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740353 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740352 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22
SRA ID: 36740351 | BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22


In [19]:
# deduplicated search - check each BioProject only once
handle = Entrez.esearch(
    db="sra",
    term="16S metagenomics gut paired same samples",
    retmax=50
)
record = Entrez.read(handle)
handle.close()

diag_ids = record["IdList"]
seen_bioprojects = set()

print("Checking unique BioProjects...\n")

for sra_id in diag_ids:
    runs_df = fetch_runs_for_bioproject(sra_id)
    if runs_df is None:
        continue
    
    bioproject = runs_df["BioProject"].iloc[0]
    
    if bioproject in seen_bioprojects:
        continue
    seen_bioprojects.add(bioproject)
    
    strategies = runs_df["LibraryStrategy"].unique()
    filters = check_hard_filters(runs_df)
    
    print(f"BioProject: {bioproject} | Strategies: {strategies} | Runs: {len(runs_df)} | passes: {filters['passes']}")
    time.sleep(0.3)

print(f"\nUnique BioProjects checked: {len(seen_bioprojects)}")

Checking unique BioProjects...

BioProject: PRJNA1205486 | Strategies: ['AMPLICON'] | Runs: 22 | passes: False
BioProject: PRJNA1168565 | Strategies: ['RNA-Seq'] | Runs: 32 | passes: False

Unique BioProjects checked: 2


In [20]:
# search BioProject database directly for projects with both data types
handle = Entrez.esearch(
    db="bioproject",
    term="gut microbiome amplicon metagenomics",
    retmax=20
)
record = Entrez.read(handle)
handle.close()

print("BioProject results:", record["Count"])
bioproject_ids = record["IdList"]
print("IDs:", bioproject_ids)

BioProject results: 35
IDs: ['1436233', '1434924', '1392086', '1390320', '1369179', '1328449', '1302757', '1283070', '1255824', '1069622', '1068958', '1060181', '1059984', '1053582', '1043851', '1043676', '979790', '918035', '860452', '832581']


In [21]:
def fetch_runs_by_bioproject_accession(bioproject_accession):
    """
    Fetch all runs for a BioProject using its accession number directly.
    
    Args:
        bioproject_accession: BioProject accession like PRJNA836960
    Returns:
        DataFrame of all runs or None if fetch failed
    """
    df = None
    
    try:
        # search SRA for all runs belonging to this BioProject
        handle = Entrez.esearch(
            db="sra",
            term=f"{bioproject_accession}[BioProject]",
            retmax=1000
        )
        record = Entrez.read(handle)
        handle.close()
        time.sleep(0.3)
        
        if record["Count"] == "0":
            return None
            
        all_ids = record["IdList"]
        ids_string = ",".join(all_ids)
        
        fetch_handle = Entrez.efetch(
            db="sra",
            id=ids_string,
            rettype="runinfo",
            retmode="text"
        )
        data = fetch_handle.read()
        fetch_handle.close()
        
        decoded = data.decode("utf-8")
        df = pd.read_csv(io.StringIO(decoded))
        df = df.dropna(subset=["Run"])
        time.sleep(0.3)
        
    except Exception as e:
        print(f"Error fetching {bioproject_accession}: {e}")
    
    return df


# now we need to convert numeric BioProject IDs to accessions
# fetch BioProject records to get accessions
print("Fetching BioProject accessions...\n")

fetch_handle = Entrez.efetch(
    db="bioproject",
    id=",".join(bioproject_ids),
    rettype="xml",
    retmode="xml"
)
bp_data = fetch_handle.read()
fetch_handle.close()

print("Raw BioProject data (first 2000 chars):")
print(bp_data[:2000])

Fetching BioProject accessions...

Raw BioProject data (first 2000 chars):
b'<?xml version="1.0" ?>\n<RecordSet><DocumentSummary uid="1436233">\n    <Project>\n        <ProjectID>\n            <ArchiveID accession="PRJEB109296" archive="EBI" id="1436233"/>\n            <CenterID center="EBI" id="0">ERP190078</CenterID>\n            <LocalID submission_id="b7ffc7ab-dcfa-4069-9983-acce27bc87c6">Technical University of Munich</LocalID>\n        </ProjectID>\n        <ProjectDescr>\n            <Name>Immunomodulatory metabolites define long-term gut microbiome recovery after allogeneic HCT and associate with improved survival and reduced relapse risk</Name>\n            <Title>Immunomodulatory Metabolites Shape Gut Microbiome Recovery and Outcomes After Allogeneic HCT</Title>\n            <Description>The intestinal microbiome influences immune recovery and long-term outcomes after alloge-neic hematopoietic stem cell transplantation (allo-SCT). While reduced bacterial diversity and depleti

In [22]:
import xml.etree.ElementTree as ET

# parse the XML to extract BioProject accessions
root = ET.fromstring(bp_data)

accessions = []
for doc in root.findall("DocumentSummary"):
    for archive_id in doc.findall(".//ArchiveID"):
        accession = archive_id.get("accession")
        if accession:
            accessions.append(accession)

print(f"Found {len(accessions)} BioProject accessions:")
for acc in accessions:
    print(f"  {acc}")

Found 20 BioProject accessions:
  PRJEB109296
  PRJEB109320
  PRJNA1392086
  PRJNA1390320
  PRJNA1369179
  PRJNA1328449
  PRJNA1302757
  PRJNA1283070
  PRJNA1255824
  PRJNA1069622
  PRJNA1068958
  PRJEB70425
  PRJEB59239
  PRJEB59610
  PRJNA1043851
  PRJNA1043676
  PRJNA979790
  PRJEB49159
  PRJEB50264
  PRJNA832581


In [24]:
import io
print("Checking hard filters on 20 BioProjects...\n")

bioproject_results = []

for accession in accessions:
    runs_df = fetch_runs_by_bioproject_accession(accession)
    
    if runs_df is None or len(runs_df) == 0:
        print(f"{accession}: no runs found")
        continue
    
    filters = check_hard_filters(runs_df)
    strategies = runs_df["LibraryStrategy"].unique().tolist()
    host = runs_df["ScientificName"].iloc[0]
    
    print(f"{accession} | {host[:30]} | strategies: {strategies} | passes: {filters['passes']}")
    
    bioproject_results.append({
        "accession": accession,
        "host": host,
        "strategies": strategies,
        "filters": filters,
        "runs": len(runs_df)
    })
    
    time.sleep(0.5)

# summary
passing = [r for r in bioproject_results if r["filters"]["passes"]]
print(f"\n--- Summary ---")
print(f"Projects checked: {len(bioproject_results)}")
print(f"Projects passing all filters: {len(passing)}")
if passing:
    print("\nPassing projects:")
    for p in passing:
        print(f"  {p['accession']} | {p['host']} | {p['strategies']}")

Checking hard filters on 20 BioProjects...

PRJEB109296 | human gut metagenome | strategies: ['WGS'] | passes: False
PRJEB109320 | chicken gut metagenome | strategies: ['AMPLICON'] | passes: False
PRJNA1392086 | gut metagenome | strategies: ['WGS'] | passes: False
PRJNA1390320 | gut metagenome | strategies: ['AMPLICON'] | passes: False
PRJNA1369179 | chicken gut metagenome | strategies: ['AMPLICON'] | passes: False
PRJNA1328449 | probiotic metagenome | strategies: ['AMPLICON'] | passes: False
PRJNA1302757 | gut metagenome | strategies: ['WGS'] | passes: False
PRJNA1283070: no runs found
PRJNA1255824 | gut metagenome | strategies: ['AMPLICON'] | passes: False
PRJNA1069622 | human gut metagenome | strategies: ['WGA'] | passes: False
PRJNA1068958 | human gut metagenome | strategies: ['AMPLICON'] | passes: False
PRJEB70425 | goat gut metagenome | strategies: ['WGS'] | passes: False
PRJEB59239: no runs found
PRJEB59610 | Canis lupus familiaris | strategies: ['AMPLICON', 'WGS'] | passes: Tru

In [25]:
# run agent on the two passing studies
for result in passing:
    study_summary = f"""
BioProject: {result['accession']}
Host organism: {result['host']}
Total runs: {result['runs']}
Library strategies: {result['strategies']}
Hard filter results: {result['filters']}
"""
    
    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=512,
        system="""You are a microbiome research assistant helping curate 
studies for the Gut Microbiome Tree of Life project (GMToL). GMToL prioritizes 
host phylogenetic diversity — studies from underrepresented animal clades are 
more valuable than additional human or mouse studies. Be concise — summarize 
in 2-3 sentences, give a value rating (Low/Medium/High/Critical), and ask 
one specific question.""",
        messages=[
            {
                "role": "user",
                "content": f"Assess this candidate study for GMToL:\n{study_summary}"
            }
        ]
    )
    
    print(f"\n{'='*50}")
    print(f"Study: {result['accession']} | Host: {result['host']}")
    print(f"{'='*50}")
    print(message.content[0].text)
    
    decision = input("\nInclude this study? (y/n/maybe): ").strip().lower()
    notes = input("Notes (press enter to skip): ").strip()
    
    all_decisions[result['accession']] = {
        "host": result['host'],
        "decision": decision,
        "notes": notes,
        "strategies": result['strategies']
    }
    
    print(f"Decision recorded.\n")
    time.sleep(0.5)

print(f"\n{'='*50}")
print(f"SESSION COMPLETE")
print(f"Total decisions in log: {len(all_decisions)}")
for bp, info in all_decisions.items(): 
    print(f"  {bp} | {info['host']} | {info['decision']}")


Study: PRJEB59610 | Host: Canis lupus familiaris
**Assessment:** This domestic dog microbiome study passes technical filters with both 16S and WGS data across 215 samples. However, dogs (Carnivora: Canidae) are already well-represented in gut microbiome literature, limiting phylogenetic novelty for GMToL.

**Value Rating:** Low

**Question:** Does this study include any wild canid samples (e.g., wolves, foxes, jackals) or focus on a unique population (e.g., feral dogs, working dogs in remote regions) that might offer novel ecological insights compared to typical pet dog studies?
Decision recorded.


Study: PRJNA1043851 | Host: Homo sapiens
**Assessment Summary:**
This study provides 72 paired-end runs with both 16S amplicon and WGS data from human samples. While the data quality metrics are favorable (paired reads, FASTQ availability, both library types), human gut microbiome is already extensively represented in GMToL, making this a low-priority addition unless it captures an underre